# Warstwa 2: Transformacja — Data Lake (Parquet)

**Cel:** Oczyszczenie surowych danych, inżynieria cech, zapis do jeziora danych w formacie Parquet  
**Wejście:** `data/raw/trending_all.csv`  
**Wyjście:** `data/parquet/trending_clean.parquet`

### Nowe cechy (feature engineering):
| Cecha | Opis |
|-------|------|
| `like_ratio` | lajki / wyświetlenia |
| `comment_ratio` | komentarze / wyświetlenia |
| `title_length` | liczba znaków w tytule |
| `title_word_count` | liczba słów w tytule |
| `publish_hour` | godzina publikacji |
| `publish_dow` | dzień tygodnia (0=Mon) |
| `publish_month` | miesiąc publikacji |
| `duration_seconds` | długość wideo w sekundach |
| `view_category` | kategoria viralności: low/medium/high/viral |
| `is_viral` | flaga: view_count > 10M |


In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

warnings.filterwarnings("ignore")

RAW_DIR     = Path("../data/raw")
PARQUET_DIR = Path("../data/parquet")
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

print("Biblioteki załadowane")

Biblioteki załadowane


In [2]:
# ── Wczytanie danych ──────────────────────────────────────────────────────────
df = pd.read_csv(RAW_DIR / "trending_all.csv", encoding="utf-8-sig")
print(f"Wczytano: {df.shape[0]:,} wierszy × {df.shape[1]} kolumn")
print(f"\nTypy danych:")
print(df.dtypes)
print(f"\nBrakujące wartości:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Wczytano: 956 wierszy × 24 kolumn

Typy danych:
video_id               str
region                 str
fetch_date             str
title                  str
description            str
channel_id             str
channel_title          str
category_id          int64
category_name          str
published_at           str
tags                   str
tag_count            int64
default_language       str
live_broadcast         str
view_count           int64
like_count           int64
comment_count        int64
comments_disabled     bool
ratings_disabled      bool
duration               str
definition             str
caption               bool
licensed_content      bool
thumbnail_url          str
dtype: object

Brakujące wartości:
description     10
tags           231
dtype: int64


In [3]:
# ── Czyszczenie — typy i wartości ─────────────────────────────────────────────
df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
df["fetch_date"]   = pd.to_datetime(df["fetch_date"],   errors="coerce")

# wypełnienie braków
df["tags"]             = df["tags"].fillna("")
df["default_language"] = df["default_language"].fillna("unknown")
df["description"]      = df["description"].fillna("")

# usuń duplikaty (ten sam film w tym samym regionie)
before = len(df)
df = df.drop_duplicates(subset=["video_id", "region"])
print(f"Usunięto duplikatów: {before - len(df)}")
print(f"Pozostało: {len(df):,} rekordów")

Usunięto duplikatów: 0
Pozostało: 956 rekordów


In [4]:
# ── Feature engineering ───────────────────────────────────────────────────────

# Ratia zaangażowania
df["like_ratio"]    = np.where(df["view_count"] > 0,
                               df["like_count"]    / df["view_count"], 0)
df["comment_ratio"] = np.where(df["view_count"] > 0,
                               df["comment_count"] / df["view_count"], 0)

# Cechy tytułu
df["title_length"]     = df["title"].str.len()
df["title_word_count"] = df["title"].str.split().str.len()
df["title_has_caps"]   = df["title"].str.contains(r'[A-ZŁŚŻŹĆÓĄĘ]{3,}', regex=True).astype(int)
df["title_has_number"] = df["title"].str.contains(r'\d', regex=True).astype(int)

# Cechy czasowe
df["publish_hour"]  = df["published_at"].dt.hour
df["publish_dow"]   = df["published_at"].dt.dayofweek   # 0=poniedziałek
df["publish_month"] = df["published_at"].dt.month
df["publish_year"]  = df["published_at"].dt.year
df["publish_dow_name"] = df["published_at"].dt.day_name()

# Dni między publikacją a pobraniem
df["days_since_publish"] = (
    df["fetch_date"] - df["published_at"].dt.tz_localize(None)
).dt.days.clip(0)

# Czas trwania w sekundach (format ISO 8601: PT1H2M3S)
def parse_duration(iso: str) -> int:
    if not isinstance(iso, str):
        return 0
    hours   = int(m.group(1)) if (m := re.search(r'(\d+)H', iso)) else 0
    minutes = int(m.group(1)) if (m := re.search(r'(\d+)M', iso)) else 0
    seconds = int(m.group(1)) if (m := re.search(r'(\d+)S', iso)) else 0
    return hours * 3600 + minutes * 60 + seconds

df["duration_seconds"] = df["duration"].apply(parse_duration)
df["duration_minutes"] = (df["duration_seconds"] / 60).round(1)

# Liczba tagów (już w danych, ale przelicz z surowych)
df["tag_count"] = df["tags"].apply(lambda x: len(x.split("|")) if x else 0)

# Kategoria wyświetleń (zmienne porządkowe)
bins   = [0, 100_000, 1_000_000, 10_000_000, float("inf")]
labels = ["low", "medium", "high", "viral"]
df["view_category"] = pd.cut(df["view_count"], bins=bins, labels=labels, right=True)
df["view_category"] = df["view_category"].astype(str)

# Flagi binarne
df["is_viral"]    = (df["view_count"] >= 10_000_000).astype(int)
df["is_hd"]       = (df["definition"] == "hd").astype(int)
df["has_caption"] = (df["caption"] == "true").astype(int)

print(f"Gotowe cechy. Wymiary: {df.shape}")
print(f"\nRozkład view_category:")
print(df["view_category"].value_counts())

Gotowe cechy. Wymiary: (956, 42)

Rozkład view_category:
view_category
low       505
medium    330
high       34
viral       5
Name: count, dtype: int64


In [5]:
# ── Walidacja ─────────────────────────────────────────────────────────────────
assert df["like_ratio"].between(0, 1).all(),    "like_ratio poza [0,1]"
assert df["comment_ratio"].between(0, 1).all(), "comment_ratio poza [0,1]"
assert df["view_count"].ge(0).all(),             "ujemne view_count"
assert df["duration_seconds"].ge(0).all(),       "ujemny czas trwania"
assert not df[["video_id", "region"]].duplicated().any(), "duplikaty"
print("Walidacja OK — brak błędów")

Walidacja OK — brak błędów


In [6]:
# ── Zapis do Parquet (Data Lake) ──────────────────────────────────────────────
out_path = PARQUET_DIR / "trending_clean.parquet"
table = pa.Table.from_pandas(df, preserve_index=False)
pq.write_table(table, out_path, compression="snappy")

print(f"Data Lake zapisany: {out_path}")
print(f"  Wiersze:   {df.shape[0]:,}")
print(f"  Kolumny:   {df.shape[1]}")
print(f"  Rozmiar:   {out_path.stat().st_size / 1024:.1f} KB")
print()

# Weryfikacja odczytu
df_check = pq.read_table(out_path).to_pandas()
print(f"Weryfikacja odczytu: {df_check.shape} ✓")

Data Lake zapisany: ..\data\parquet\trending_clean.parquet
  Wiersze:   956
  Kolumny:   42
  Rozmiar:   442.6 KB

Weryfikacja odczytu: (956, 42) ✓


In [7]:
# ── Podgląd finalnych danych ──────────────────────────────────────────────────
print("Przykładowe rekordy:")
cols_preview = ["title", "region", "view_count", "like_ratio",
                "category_name", "view_category", "publish_hour", "duration_minutes"]
df[cols_preview].head(10)

Przykładowe rekordy:


,title,region,view_count,like_ratio,category_name,view_category,publish_hour,duration_minutes
0,"HELLFIELD, SENTINO - Mallorca (Official Video)",PL,1157591,0.018889,Music,high,16,3.2
1,Vought Rising - First Look | Prime Video,PL,1480958,0.041302,Entertainment,high,14,1.6
2,♪ LUCZEK - SERDUSZKO ❤️ feat. PIMPEK [OFICIALN...,PL,1151476,0.015340,Gaming,high,13,2.4
3,"Shakira, Burna Boy - Dai Dai (Official Video)",PL,27388277,0.053602,Howto & Style,viral,16,4.0
4,ZROBIŁEM SKANER na MOICH PRZYJACIÓŁ na Wojanow...,PL,189792,0.023837,Gaming,medium,12,21.1
5,VKIE - NIE NA PRÓŻNO [🎥:XAWITO],PL,283661,0.054318,Music,medium,10,2.7
6,WIEŻA CO BYŚ WOLAŁ CHŁOPACY VS DZIEWCZYNY w Ro...,PL,202626,0.025816,Gaming,medium,11,18.5
7,NIE ŚPIĘ,PL,523441,0.002858,Music,medium,0,3.2
8,SAGI VS CAŁA ANARCHIA - ANARCHIA SMP 3,PL,191921,0.051761,Gaming,medium,20,143.5
9,Mała Armia Janosika śpiewa hymn Wisły Kraków n...,PL,107628,0.026257,Music,medium,15,2.2
